# Alumni RAG Agent - System Presentation & Implementation

## 1. Project Overview

This project implements an agentic system for CMU Africa Alumni Tracking and Support. It uses a **Retrieval-Augmented Generation (RAG)** architecture with a reasoning loop to perform complex tasks like identifying career changes and triggering personalized outreach.

### Core Capabilities
1. **Retrieval Module**: Semantic search over alumni profiles using MongoDB Atlas Vector Search.
2. **Tool-Calling Module**: Capability to perform external actions (LinkedIn scraping, Email, Surveys).
3. **Verification Module**: Groundedness scoring to ensure responses are factually supported by retrieved data.
4. **Automated Discovery**: New capability to auto-discover alumni profiles via Google Search.

## 2. System Architecture

### Data Pipeline
The system ingests data from multiple sources into a centralized Vector Store:
```
LinkedIn Scraping / Manual CSV → Python Ingestion → MongoDB Storage → Embedding Generation → Vector Index
```

### Agent Loop (ReAct Pattern)
The agent follows a **Reasoning and Acting (ReAct)** loop:
1. **OBSERVE**: Analyzes the user query and current context.
2. **REASON**: Decides what information is missing or what action is needed.
3. **DECIDE**: Chooses a tool (Retrieval, Email, Search, etc.).
4. **ACT**: Executes the tool and captures the output.
5. **VERIFY**: Checks the final response against the retrieved evidence.

## 3. Environment Configuration

The system requires the following keys to be set in `.env`:

In [ ]:
import os
import sys
from dotenv import load_dotenv

load_dotenv()

# Enable LangSmith Tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "alumni-rag-agent-hw2"

print("Configuration Status:")
print(f"  MongoDB URI:      {'✓ Ready' if os.environ.get('MONGODB_URI') else '✗ Missing'}")
print(f"  OpenAI API Key:   {'✓ Ready' if os.environ.get('OPENAI_API_KEY') else '✗ Missing'}")
print(f"  Google API Key:   {'✓ Ready' if os.environ.get('GOOGLE_API_KEY') else '✗ Missing (Optional)'}")

## 4. Module 1: Retrieval Module (MongoDB Atlas)

The retrieval module connects to MongoDB Atlas Vector Search. It stores alumni profiles as embeddings (1536 dimensions) along with metadata for filtering (Year, Program, etc.).

**Schema:**
```python
{
    "content": "John Doe graduated from CMU Africa in 2023...",
    "embedding": [0.0123, ...],
    "metadata": {"alumni_id": "A2023-001", "curr_job": "Data Engineer"}
}
```

In [ ]:
from src.agent import AlumniAgent, SAMPLE_ALUMNI

# Initialize the Agent (which initializes the Vector Store)
agent = AlumniAgent()

# Ingest sample data for demonstration
print(f"Ingesting {len(SAMPLE_ALUMNI)} sample profiles...")
agent.ingest_alumni(SAMPLE_ALUMNI)

# Test Search
print("\nTest Retrieval (Query: 'fintech'):")
results = agent.search("fintech", k=2)
for doc in results:
    print(f"  - Found: {doc.metadata['name']} ({doc.metadata['company']})")

## 5. Module 2: Tool-Calling Module

The agent has access to specific tools to perform actions. The `ReActAgent` chooses these dynamically.

| Tool | Purpose |
|------|---------|
| `linkedin_scraper` | Monitors profiles for career changes |
| `email_sender` | Sends personalized emails (Dry Run mode) |
| `survey_tool` | Creates and sends Google Forms |
| `linkedin_discovery` | **(New)** Auto-finds profiles via Google Search |

In [ ]:
# Demonstrate Automated Discovery Tool
if os.environ.get("GOOGLE_API_KEY"):
    print("Demonstrating Automated Discovery (Google Search -> Scrape -> Ingest):\n")
    new_profiles = agent.discover_and_ingest(program="MSIT", year=2023)
else:
    print("Google API Key not set - skipping discovery demo.")
    print("You can still use the manual tools below.")

## 6. Module 3: Verification Module

To prevent hallucinations, the system uses a **Groundedness Scorer**.

**Logic:**
1. Extract factual claims from the agent's response.
2. Check each claim against the retrieved source documents.
3. Calculate Score = (Verified Claims / Total Claims).
4. If Score < Threshold, warning caveats are added.

## 7. End-to-End System Demo

We will now run a complex query that requires the agent to:
1. **Search** the database for specific alumni.
2. **Reason** about who fits the criteria.
3. **Act** by sending an email.
4. **Verify** the action.

In [ ]:
QUERY = "Find alumni who work in fintech and send them a check-in email"

print(f"Executing Query: '{QUERY}'...\n")
result = agent.run(QUERY)

print("\n" + "="*60)
print("FINAL RESPONSE")
print("="*60)
print(result["response"])

### Verification Report Details
Below is the breakdown of the verification for the above response.

In [ ]:
v = result["verification"]
print(f"Groundedness Score: {v.score:.2f} ({v.confidence})")
print("Claims Verified:")
for c in v.claims:
    print(f"  [{'✓' if c.verified else '✗'}] {c.claim}")

## 8. Conclusion

This system demonstrates a complete Agentic RAG workflow. The separation of concerns (Retrieval, Tools, Verification) allows for a robust and scalable alumni support system.

Full execution traces are available in **LangSmith**.